## Supplemental EDA

Implemented by Mary

A clearer view of all data used as supplement to our results analysis and data decision-making. 

In [ ]:
# %load_ext autoreload
# %autoreload 2

# Setup and data acquisition
import sys
sys.path.append('./modules')
import get_data
import pandas as pd
import seaborn as sns

# Download raw data files
datafiles = [
    { 'url': 'https://drive.google.com/uc?export=download&id=1NJzzTgyM5pExxJJS6YQav2JGtYhBoGGI', 'filename':'mental_health_physical_activity.csv'},
    { 'url': 'https://drive.google.com/uc?export=download&id=1fGvIjW5TaZfea9buV7KzgNEtaCbaFePQ', 'filename':'diabetes.csv'}
]
get_data.get_raw(datafiles, destination_directory='data/00-raw/')

# Process Diabetes BRFSS 2015 dataset
diabetes = pd.read_csv("data/00-raw/diabetes.csv")
print("Diabetes dataset shape:", diabetes.shape)
print("Missing values:\n", diabetes.isna().sum())

# Drop irrelevant columns
cols_to_drop = ["Fruits", "Veggies", "CholCheck", "Smoker", "Stroke", 
                "HeartDiseaseorAttack", "HvyAlcoholConsump", "AnyHealthcare", 
                "NoDocbcCost", "Education", "Income"]
clean_diabetes = diabetes.drop(columns=cols_to_drop)

# Transform age categories
age_bins = {1: "18-24", 2: "25-29", 3: "30-34", 4: "35-39", 5: "40-44", 
            6: "45-49", 7: "50-54", 8: "55-59", 9: "60-64", 10: "65-69", 
            11: "70-74", 12: "75-80", 13: "80 or older"}
clean_diabetes["Age"] = clean_diabetes["Age"].map(age_bins)

# Summary statistics
print("\nDiabetes dataset statistics:")
print(clean_diabetes.describe())

# Visualization
sns.histplot(x="BMI", hue="Diabetes_binary", data=clean_diabetes, bins=30)

# Save cleaned diabetes dataset
clean_diabetes.to_csv('data/02-processed/diabetes_clean.csv', index=False)

# Process Mental Health x Physical Activity dataset
activity_mental = pd.read_csv('data/00-raw/mental_health_physical_activity.csv')
print("\nMental health dataset columns:", activity_mental.columns.tolist())
print("Duplicate IDs:", activity_mental['ID'].duplicated().sum())
print("Mental health dataset shape:", activity_mental.shape)

# Check missing values
print("Missing values:\n", activity_mental.isna().sum())

# Clean mental health dataset
clean_am = activity_mental.dropna(subset='Exercise_Type').drop(columns='Notes')

# Examine Mental_Health_Score
print("\nMental Health Score statistics:")
print(clean_am['Mental_Health_Score'].describe())

# Drop calculated score, keep self-reported values
clean_am_sr = clean_am.drop(columns='Mental_Health_Score')

# Summary statistics for key variables
print("\nKey variables statistics:")
print(clean_am_sr[['Age', 'Gender', 'Exercise_Frequency', 'Exercise_Duration',
                   'Stress_Level', 'Anxiety_Level', 'Depression_Level', 
                   'Happiness_Level']].describe())

# Save cleaned mental health dataset
clean_am_sr.to_csv('data/02-processed/mental_health_physical_activity_clean.csv', index=False)

print("\nData processing complete. Cleaned datasets saved to data/02-processed/")

In [ ]:
# mental health notebook
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Read the cleaned data
clean_am_sr = pd.read_csv('data/02-processed/mental_health_physical_activity_clean.csv')

# Calculate weekly exercise minutes
clean_am_sr['Weekly_Exercise_Minutes'] = (
    clean_am_sr['Exercise_Frequency'] * clean_am_sr['Exercise_Duration']
)

# Create exercise categories based on WHO guidelines
def categorize_exercise(minutes):
    if minutes < 60:
        return 'Minimal (<60 min)'
    elif minutes < 150:
        return 'Insufficient (60-149 min)'
    elif minutes < 300:
        return 'Adequate (150-299 min)'
    else:
        return 'High (300+ min)'

clean_am_sr['Exercise_Category'] = clean_am_sr['Weekly_Exercise_Minutes'].apply(categorize_exercise)

# Create age groups that match BRFSS 5-year bins
def assign_age_group(age):
    if age < 25:
        return '18-24'
    elif age < 30:
        return '25-29'
    elif age < 35:
        return '30-34'
    elif age < 40:
        return '35-39'
    elif age < 45:
        return '40-44'
    elif age < 50:
        return '45-49'
    elif age < 55:
        return '50-54'
    elif age < 60:
        return '55-59'
    elif age < 65:
        return '60-64'
    elif age < 70:
        return '65-69'
    elif age < 75:
        return '70-74'
    elif age < 80:
        return '75-79'
    else:
        return '80 or older'

clean_am_sr['Age_Group'] = clean_am_sr['Age'].apply(assign_age_group)

# Create composite mental health score (reverse-coded negative indicators)
# Higher score = better mental health
clean_am_sr['Mental_Health_Composite'] = (
    clean_am_sr['Happiness_Level'] - 
    (clean_am_sr['Stress_Level'] + clean_am_sr['Anxiety_Level'] + clean_am_sr['Depression_Level']) / 3
)

# Save updated dataset
clean_am_sr.to_csv('data/02-processed/mental_health_with_weekly_exercise.csv', index=False)

In [ ]:
# Aggregate mental health dataset by age group and gender
mh_demo_summary = clean_am_sr.groupby(['Age_Group', 'Gender']).agg({
    'Weekly_Exercise_Minutes': ['mean', 'median', 'std', 'count'],
    'Mental_Health_Composite': 'mean',
    'Happiness_Level': 'mean',
    'Stress_Level': 'mean',
    'Anxiety_Level': 'mean',
    'Depression_Level': 'mean',
    'Exercise_Category': lambda x: x.value_counts().to_dict()  # Distribution of categories
}).round(2)

# Flatten column names
mh_demo_summary.columns = ['_'.join(col).strip() for col in mh_demo_summary.columns.values]
mh_demo_summary = mh_demo_summary.reset_index()

# Create a simpler version for comparison
mh_comparison = clean_am_sr.groupby(['Age_Group', 'Gender']).agg({
    'Weekly_Exercise_Minutes': 'mean',
    'Mental_Health_Composite': 'mean',
    'Happiness_Level': 'mean',
    'Stress_Level': 'mean'
}).round(2).reset_index()

print("Mental Health Dataset - Demographic Summary:")
print(mh_comparison)
mh_comparison.to_csv('data/03-summary/mh_demographic_summary.csv', index=False)

In [ ]:
# Aggregate mental health dataset by age group and gender
mh_demo_summary = clean_am_sr.groupby(['Age_Group', 'Gender']).agg({
    'Weekly_Exercise_Minutes': ['mean', 'median', 'std', 'count'],
    'Mental_Health_Composite': 'mean',
    'Happiness_Level': 'mean',
    'Stress_Level': 'mean',
    'Anxiety_Level': 'mean',
    'Depression_Level': 'mean',
    'Exercise_Category': lambda x: x.value_counts().to_dict()  # Distribution of categories
}).round(2)

# Flatten column names
mh_demo_summary.columns = ['_'.join(col).strip() for col in mh_demo_summary.columns.values]
mh_demo_summary = mh_demo_summary.reset_index()

# Create a simpler version for comparison
mh_comparison = clean_am_sr.groupby(['Age_Group', 'Gender']).agg({
    'Weekly_Exercise_Minutes': 'mean',
    'Mental_Health_Composite': 'mean',
    'Happiness_Level': 'mean',
    'Stress_Level': 'mean'
}).round(2).reset_index()

print("Mental Health Dataset - Demographic Summary:")
print(mh_comparison)
mh_comparison.to_csv('data/03-summary/mh_demographic_summary.csv', index=False)

In [ ]:
# Read cleaned BRFSS data
clean_diabetes = pd.read_csv('data/02-processed/diabetes_clean.csv')

# Create gender labels
clean_diabetes['Gender'] = clean_diabetes['Sex'].map({0: 'Female', 1: 'Male'})

# Create BMI categories for obesity analysis
def bmi_category(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

clean_diabetes['BMI_Category'] = clean_diabetes['BMI'].apply(bmi_category)

# Create obesity binary (BMI >= 30)
clean_diabetes['Obesity'] = (clean_diabetes['BMI'] >= 30).astype(int)

# Aggregate BRFSS by age group and gender
brfss_demo_summary = clean_diabetes.groupby(['Age', 'Gender']).agg({
    'PhysActivity': 'mean',  # Proportion physically active
    'BMI': 'mean',
    'Obesity': 'mean',  # Obesity rate
    'Diabetes_binary': 'mean',  # Diabetes rate
    'MentHlth': 'mean',  # Average poor mental health days
    'HighBP': 'mean',  # High blood pressure rate
    'HighChol': 'mean',  # High cholesterol rate
    'GenHlth': 'mean',  # General health (lower is better)
    'PhysHlth': 'mean',  # Poor physical health days
    'DiffWalk': 'mean',  # Difficulty walking rate
    'Sex': 'count'  # Count in each group
}).round(3)

# Rename count column
brfss_demo_summary = brfss_demo_summary.rename(columns={'Sex': 'Count'})
brfss_demo_summary = brfss_demo_summary.reset_index()

print("\nBRFSS Dataset - Demographic Summary:")
print(brfss_demo_summary.head())
brfss_demo_summary.to_csv('data/03-summary/brfss_demographic_summary.csv', index=False)

# Focus on your specific hypothesis groups
# Men 18-30
men_18_30 = clean_diabetes[
    (clean_diabetes['Gender'] == 'Male') & 
    (clean_diabetes['Age'].isin(['18-24', '25-29', '30-34']))
]

# Women 40+
women_40_plus = clean_diabetes[
    (clean_diabetes['Gender'] == 'Female') & 
    (clean_diabetes['Age'].isin(['40-44', '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75-80', '80 or older']))
]

print("\n--- Hypothesis Testing Groups ---")
print(f"Men 18-34: {len(men_18_30)} respondents")
print(f"Women 40+: {len(women_40_plus)} respondents")

# Summary stats for hypothesis groups
hypothesis_summary = pd.DataFrame({
    'Group': ['Men 18-34', 'Women 40+'],
    'Physically Active %': [
        men_18_30['PhysActivity'].mean() * 100,
        women_40_plus['PhysActivity'].mean() * 100
    ],
    'Obesity Rate %': [
        (men_18_30['BMI'] >= 30).mean() * 100,
        (women_40_plus['BMI'] >= 30).mean() * 100
    ],
    'Diabetes Rate %': [
        men_18_30['Diabetes_binary'].mean() * 100,
        women_40_plus['Diabetes_binary'].mean() * 100
    ],
    'Poor Mental Health Days (avg)': [
        men_18_30['MentHlth'].mean(),
        women_40_plus['MentHlth'].mean()
    ]
})

print("\nHypothesis Groups Comparison:")
print(hypothesis_summary)
hypothesis_summary.to_csv('data/03-summary/hypothesis_groups_comparison.csv', index=False)

In [ ]:
# Create comparison of exercise patterns by demographic groups

# Get exercise minutes from mental health dataset
mh_exercise = clean_am_sr.groupby(['Age_Group', 'Gender']).agg({
    'Weekly_Exercise_Minutes': 'mean',
    'Exercise_Frequency': 'mean',
    'Exercise_Duration': 'mean'
}).round(2).reset_index()

# Get binary activity from BRFSS dataset
brfss_activity = clean_diabetes.groupby(['Age', 'Gender']).agg({
    'PhysActivity': 'mean'
}).round(3).reset_index()
brfss_activity['PhysActivity_Percent'] = brfss_activity['PhysActivity'] * 100

# Create comparison table
# First, align age groups (note: BRFSS has 13 groups, MH has fewer)
exercise_comparison = pd.DataFrame()

# For mental health dataset, calculate what percentage meet WHO guidelines
mh_who = clean_am_sr.copy()
mh_who['Meets_WHO'] = mh_who['Weekly_Exercise_Minutes'] >= 150

who_summary = mh_who.groupby(['Age_Group', 'Gender']).agg({
    'Meets_WHO': 'mean',
    'Weekly_Exercise_Minutes': 'mean'
}).round(3).reset_index()
who_summary['Meets_WHO_Percent'] = who_summary['Meets_WHO'] * 100

print("\n--- Exercise Patterns Comparison ---")
print("\nMental Health Dataset - Weekly Minutes by Demographics:")
print(mh_exercise)

print("\nMental Health Dataset - % Meeting WHO Guidelines (150+ min/week):")
print(who_summary[['Age_Group', 'Gender', 'Meets_WHO_Percent', 'Weekly_Exercise_Minutes']])

print("\nBRFSS Dataset - % Physically Active (any exercise in past 30 days):")
print(brfss_activity[brfss_activity['PhysActivity_Percent'] > 0].head(10))

# Save comparisons
mh_exercise.to_csv('data/03-summary/mh_exercise_minutes_by_demo.csv', index=False)
who_summary.to_csv('data/03-summary/who_guidelines_met.csv', index=False)
brfss_activity.to_csv('data/03-summary/brfss_activity_by_demo.csv', index=False)

In [ ]:
# Correlation analysis for mental health dataset
mh_corr_vars = ['Weekly_Exercise_Minutes', 'Mental_Health_Composite', 
                'Happiness_Level', 'Stress_Level', 'Anxiety_Level', 
                'Depression_Level', 'Age', 'Sleep_Hours', 'Diet_Quality']

mh_corr = clean_am_sr[mh_corr_vars].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(mh_corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix: Exercise & Mental Health')
plt.tight_layout()
plt.savefig('data/03-summary/mh_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# Exercise categories vs mental health outcomes
mh_category_summary = clean_am_sr.groupby('Exercise_Category').agg({
    'Mental_Health_Composite': 'mean',
    'Happiness_Level': 'mean',
    'Stress_Level': 'mean',
    'Anxiety_Level': 'mean',
    'Depression_Level': 'mean',
    'Weekly_Exercise_Minutes': 'count'
}).round(2).rename(columns={'Weekly_Exercise_Minutes': 'Count'})

print("\nMental Health Outcomes by Exercise Category:")
print(mh_category_summary)
mh_category_summary.to_csv('data/03-summary/mh_by_exercise_category.csv')

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Happiness by exercise minutes
axes[0,0].scatter(clean_am_sr['Weekly_Exercise_Minutes'], clean_am_sr['Happiness_Level'], alpha=0.5)
axes[0,0].set_xlabel('Weekly Exercise Minutes')
axes[0,0].set_ylabel('Happiness Level (1-10)')
axes[0,0].set_title('Happiness vs Exercise Minutes')

# Stress by exercise minutes
axes[0,1].scatter(clean_am_sr['Weekly_Exercise_Minutes'], clean_am_sr['Stress_Level'], alpha=0.5, color='red')
axes[0,1].set_xlabel('Weekly Exercise Minutes')
axes[0,1].set_ylabel('Stress Level (1-10)')
axes[0,1].set_title('Stress vs Exercise Minutes')

# Mental Health Composite by exercise category
mh_category_summary_sorted = mh_category_summary.reset_index().sort_values('Mental_Health_Composite')
axes[1,0].barh(mh_category_summary_sorted['Exercise_Category'], 
               mh_category_summary_sorted['Mental_Health_Composite'])
axes[1,0].set_xlabel('Mental Health Composite Score')
axes[1,0].set_title('Mental Health by Exercise Category')

# Exercise minutes distribution
axes[1,1].hist(clean_am_sr['Weekly_Exercise_Minutes'], bins=20, edgecolor='black')
axes[1,1].axvline(150, color='red', linestyle='--', label='WHO Guidelines (150 min)')
axes[1,1].set_xlabel('Weekly Exercise Minutes')
axes[1,1].set_ylabel('Frequency')
axes[1,1].set_title('Distribution of Weekly Exercise Minutes')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('data/03-summary/mh_exercise_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation analysis for BRFSS
brfss_corr_vars = ['BMI', 'Diabetes_binary', 'MentHlth', 'PhysHlth', 
                   'HighBP', 'HighChol', 'PhysActivity', 'GenHlth']

brfss_corr = clean_diabetes[brfss_corr_vars].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(brfss_corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix: Health Indicators (BRFSS 2015)')
plt.tight_layout()
plt.savefig('data/03-summary/brfss_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# Compare active vs inactive individuals
active_vs_inactive = clean_diabetes.groupby('PhysActivity').agg({
    'BMI': 'mean',
    'Obesity': 'mean',
    'Diabetes_binary': 'mean',
    'MentHlth': 'mean',
    'PhysHlth': 'mean',
    'GenHlth': 'mean',
    'HighBP': 'mean',
    'HighChol': 'mean'
}).round(3)

active_vs_inactive.index = ['Inactive (no exercise)', 'Active (any exercise)']
print("\nHealth Outcomes: Active vs Inactive Individuals:")
print(active_vs_inactive)
active_vs_inactive.to_csv('data/03-summary/active_vs_inactive_comparison.csv')

# Age trends
age_trends = clean_diabetes.groupby('Age').agg({
    'PhysActivity': 'mean',
    'BMI': 'mean',
    'Obesity': 'mean',
    'Diabetes_binary': 'mean',
    'MentHlth': 'mean'
}).reset_index()

# Plot age trends
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Physical activity by age
axes[0,0].plot(age_trends['Age'], age_trends['PhysActivity'] * 100, marker='o')
axes[0,0].set_xlabel('Age Group')
axes[0,0].set_ylabel('% Physically Active')
axes[0,0].set_title('Physical Activity by Age')
axes[0,0].tick_params(axis='x', rotation=45)

# Obesity by age
axes[0,1].plot(age_trends['Age'], age_trends['Obesity'] * 100, marker='o', color='orange')
axes[0,1].set_xlabel('Age Group')
axes[0,1].set_ylabel('% Obese')
axes[0,1].set_title('Obesity Rate by Age')
axes[0,1].tick_params(axis='x', rotation=45)

# Diabetes by age
axes[1,0].plot(age_trends['Age'], age_trends['Diabetes_binary'] * 100, marker='o', color='red')
axes[1,0].set_xlabel('Age Group')
axes[1,0].set_ylabel('% with Diabetes/Prediabetes')
axes[1,0].set_title('Diabetes Rate by Age')
axes[1,0].tick_params(axis='x', rotation=45)

# Mental health by age
axes[1,1].plot(age_trends['Age'], age_trends['MentHlth'], marker='o', color='green')
axes[1,1].set_xlabel('Age Group')
axes[1,1].set_ylabel('Poor Mental Health Days (avg)')
axes[1,1].set_title('Mental Health by Age')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('data/03-summary/brfss_age_trends.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Create a comprehensive comparison table
synthesis = pd.DataFrame({
    'Metric': [
        'Sample Size',
        'Exercise Measure',
        'Population',
        'Year',
        'Key Finding: Exercise & Mental Health',
        'Key Finding: Exercise & Obesity',
        'Key Finding: Exercise & Diabetes',
        'Gender Pattern',
        'Age Pattern'
    ],
    'Mental Health Dataset': [
        f"{len(clean_am_sr)}",
        'Continuous (minutes/week)',
        'Convenience sample',
        'Unknown (recent?)',
        f"Correlation: {mh_corr.loc['Weekly_Exercise_Minutes', 'Mental_Health_Composite']:.2f}",
        'N/A - No BMI data',
        'N/A - No diabetes data',
        f"Men: {clean_am_sr[clean_am_sr['Gender']=='Male']['Weekly_Exercise_Minutes'].mean():.0f} min, Women: {clean_am_sr[clean_am_sr['Gender']=='Female']['Weekly_Exercise_Minutes'].mean():.0f} min",
        'Check age groups'
    ],
    'BRFSS Dataset': [
        f"{len(clean_diabetes)}",
        'Binary (any activity)',
        'Representative U.S. sample',
        '2015',
        f"Active: {clean_diabetes[clean_diabetes['PhysActivity']==1]['MentHlth'].mean():.1f} poor days, Inactive: {clean_diabetes[clean_diabetes['PhysActivity']==0]['MentHlth'].mean():.1f} poor days",
        f"Active BMI: {clean_diabetes[clean_diabetes['PhysActivity']==1]['BMI'].mean():.1f}, Inactive BMI: {clean_diabetes[clean_diabetes['PhysActivity']==0]['BMI'].mean():.1f}",
        f"Active diabetes rate: {clean_diabetes[clean_diabetes['PhysActivity']==1]['Diabetes_binary'].mean()*100:.1f}%, Inactive: {clean_diabetes[clean_diabetes['PhysActivity']==0]['Diabetes_binary'].mean()*100:.1f}%",
        f"Men active: {clean_diabetes[clean_diabetes['Sex']==1]['PhysActivity'].mean()*100:.0f}%, Women active: {clean_diabetes[clean_diabetes['Sex']==0]['PhysActivity'].mean()*100:.0f}%",
        'Physical activity declines with age, health conditions increase'
    ]
})

print("\n" + "="*80)
print("SYNTHESIS OF FINDINGS ACROSS DATASETS")
print("="*80)
print(synthesis.to_string(index=False))

# Test your specific hypotheses
print("\n" + "="*80)
print("HYPOTHESIS TESTING")
print("="*80)

# Hypothesis 1: Strong negative correlation between exercise and obesity/diabetes
print("\nHypothesis 1: Strong negative correlation between exercise and obesity/diabetes")
print(f"BRFSS - Obesity rate: Active={active_vs_inactive.loc['Active (any exercise)', 'Obesity']*100:.1f}%, Inactive={active_vs_inactive.loc['Inactive (no exercise)', 'Obesity']*100:.1f}%")
print(f"BRFSS - Diabetes rate: Active={active_vs_inactive.loc['Active (any exercise)', 'Diabetes_binary']*100:.1f}%, Inactive={active_vs_inactive.loc['Inactive (no exercise)', 'Diabetes_binary']*100:.1f}%")
print(f"Difference: Obesity {-active_vs_inactive.diff().loc['Active (any exercise)', 'Obesity']*100:.1f} percentage points, Diabetes {-active_vs_inactive.diff().loc['Active (any exercise)', 'Diabetes_binary']*100:.1f} percentage points")

# Hypothesis 2: Positive correlation between exercise and mental health
print("\nHypothesis 2: Positive correlation between exercise and mental health")
print(f"Mental Health Dataset - Correlation (exercise × mental health): {mh_corr.loc['Weekly_Exercise_Minutes', 'Mental_Health_Composite']:.2f}")
print(f"BRFSS - Poor mental health days: Active={active_vs_inactive.loc['Active (any exercise)', 'MentHlth']:.1f} days, Inactive={active_vs_inactive.loc['Inactive (no exercise)', 'MentHlth']:.1f} days")

# Hypothesis 3: Age/gender variations
print("\nHypothesis 3: Age and gender variations")
print(f"Men 18-34: {len(men_18_30)} respondents, Active: {men_18_30['PhysActivity'].mean()*100:.1f}%, Obesity: {(men_18_30['BMI']>=30).mean()*100:.1f}%, Diabetes: {men_18_30['Diabetes_binary'].mean()*100:.1f}%")
print(f"Women 40+: {len(women_40_plus)} respondents, Active: {women_40_plus['PhysActivity'].mean()*100:.1f}%, Obesity: {(women_40_plus['BMI']>=30).mean()*100:.1f}%, Diabetes: {women_40_plus['Diabetes_binary'].mean()*100:.1f}%")

# Save synthesis
synthesis.to_csv('data/03-summary/cross_dataset_synthesis.csv', index=False)

In [ ]:
import os

# Create necessary directories if they don't exist
os.makedirs('data/03-summary', exist_ok=True)
os.makedirs('figures', exist_ok=True)